# 리포트 디렉터리 점검 — 연구 자산의 구성과 다음 수정 우선순위

> ### 한 일
> **디렉터리 구성·핵심 처리 코드·문서·기존 검사를 대조하고 합성 입력으로 계산을 확인했다.**

### 결과
1. 선택한 검사 6 [^1]개 중 6 [^2]개가 종료 코드 기준으로 통과했다.
2. 강한 셀을 포함한 합성 입력에서 CFAR 누적 정밀도 차이에 따른 추가 검출을 재현했다.
3. 현재 거리 표시와 배열 이득 실험에는 정답 위치·이상적 조향 가정이 들어 있다.
4. 새 실험의 출발점은 입력 계약·환경 재현·발간 범위를 함께 정리하는 것이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 파일·문서 | 파일 목록과 Python 구문을 검사하고 핵심 경로를 선별해서 읽었다. |
| 계산 | CPU 합성 반례와 대조 입력을 사용했다. 실측과 전체 GPU 실험 재실행은 이번 범위 밖이다. |

### 재현

```bash
/workspace/.venvs/py312/bin/python benchmark/review_directory_0915.py
```

| | |
|---|---|
| 출력 | `outputs/directory_review_0915.json` |
| 소요 | CPU 단일 코어; 검사별 소요는 원장 참조 |

---

## 구성과 강점

| 위치 | 역할 |
|---|---|
| `src/` | 산란 커널·신호 처리·보고서 빌더 |
| `benchmark/` | 실험·합성 반례·원장 생성·검사 |
| `runners/` | 작업 목록·supervisor·솔버 버전 장부 |
| `outputs/` | 수치 원장·그림·자세별 shard |
| `reports/` | 독자용 notebook과 생성용 조각 |
| `docs/` | 규약·재현 안내·진행 및 검토 기록 |
| `assets/` · `vendor/` · `work/` | 형상 자산·설치 wheel·중간 작업 |

원장 → 빌더 → notebook의 경로, 철회 기록, 입력 유한값 검사, 솔버 버전 꼬리표는 유지할 가치가 있다.

권 색인에는 본편 13 [^3]권·별편 7 [^4]편·notebook 33 [^5]개가 기록돼 있다.
파일 수는 작업 중 스냅샷이다. 내용 검토는 핵심 코드와 문서를 선별한 범위다.

## 우선 수정: CFAR 누적 정밀도

합성 RD 크기 64 [^6]×32 [^7], 명목 Pfa 0.0001 [^8], 강한 셀 [32 [^9], 2 [^10]], 배경 단위전력 대비 100 [^11] dB 조건이다.
중앙 guard 행을 제외한 검출 수는 float32 583 [^12]셀, float64 0 [^13]셀이다.

관찰: `src/detection_gpu.py`는 누적합을 입력 dtype으로 계산한다. 강한 셀 주변 누적합의 차를 구하는 과정이 점검 대상이다.
권고: 높은 정밀도의 누적합과 직접 training-window 합을 대조하는 회귀검사를 먼저 마련한다.

범위: CPU 합성 입력의 수치 반례다. 기존 원장 Pd/Pfa에 미친 영향은 실제 IQ별 재평가가 필요하다.

## 추적 확장 전: 정답과 수신 처리를 분리

동일 IQ의 정답 메타데이터를 200 [^14] m에서 300 [^15] m로 바꾸면 표시 축이 100 [^16] m 이동한다.
RD 배열은 같은 상태다. `Precomputed`의 거리축은 표적 정답에 맞춘 시각화다.

`gpu_montecarlo`는 steering vector 길이로 이상적인 배열 이득을 적용한다. 실제 방향 추정·조향 오차는 별도 평가 항목이다.
권고: delay 기준과 수신 시각으로 거리축을 정하고, 표적 truth는 평가 단계에서 결합한다.

`src/passive_process.py` 머리말은 다중프레임 판정과 Kalman을 향후 범위로 명시한다. 탐지 결과를 추적 완성도로 읽을 때 이 구분이 필요하다.

## 파형 확장 전: 프레임별 기준신호 계약

`range_doppler`와 `rd_batch`는 첫 프레임 기준신호를 여러 프레임에 재사용한다.
독립 QPSK 32 [^17]프레임·길이 128 [^18]표본·지연 7 [^19]표본·Doppler offset 5 [^20]빈 합성 입력에서 정답 셀 피크는 프레임별 정합 대비 -37.78 [^21] dB다.
같은 프레임 반복 대조 입력에서는 0.00 [^22] dB다.
해석: payload가 프레임마다 달라지는 실험에는 프레임별 reference 처리 계약이 필요하다. 표준 파형이나 RF 성능을 측정한 값은 아니다.

`waveforms_sionna.crosscheck`는 같은 자작 resource grid를 두 OFDM 변조기에 넣는다. 변조 계산의 일치와 전체 표준 파형 적합성은 검사 범위가 다르다.

## 환경·문서의 현재 상태

설치 메타데이터의 Sionna RT는 `2.1.0`이고, `runners/SOLVER_BUILDS.json`에 그 조합이 기록돼 있다.
`README.md`와 `benchmark/README.md`의 도입부는 이전 솔버 버전을 안내한다. 과거 결과 환경과 새 실험 환경을 나눠 표시하는 것이 좋다.

조사한 setup 파일명 범위에서는 주 프로젝트의 dependency manifest와 lockfile이 검색되지 않았다. `jihyuck/po_mdoppler/requirements.txt`는 별도 코드용이다.
`g.xml`은 지면 장면을 절대경로로 참조한다. 이름·용도·생성 경로를 문서화하고 다른 checkout에서의 사용 여부를 확인할 필요가 있다.

`docs/RESUME.md`는 과거 날짜를 안내하고 최근 일일 resume에도 이전 운영 상태가 남아 있다. 현재 상태를 읽는 입구를 한 곳으로 정하는 것이 좋다.

## 통과한 검사와 남는 범위

Python 구문 검사 대상은 749 [^23]파일, 구문 오류는 0 [^24]건이다.
아래 종료 코드는 검사 범위에서의 결과다. 물리적 타당성이나 전체 빌드 성공과는 별도다.
| 검사 | 종료 코드 |
|---|---|
| `check_reader_gate` | 0 [^25] |
| `check_report_links` | 0 [^26] |
| `check_retracted` | 0 [^27] |
| `check_stale_titles` | 0 [^28] |
| `check_row_pointers` | 0 [^29] |
| `check_new_file_rules` | 0 [^30] |

`check_new_file_rules`는 기준선 이후 새 항목을 차단한다. 기존 기준선 항목도 검사 출력에 남는다.

`check_reader_gate`도 알려진 미차단 표시 패턴을 출력하며 통과한다. 통과 코드만 요약하면 이 범위가 사라진다.

`docs/MESH_GALLERY_STALE_0914.md`에는 표적 보고서 빌더의 갤러리 원장 불일치가 기록돼 있다. 이번에는 그 빌더를 실행하지 않았으므로 현재 실패를 재확인한 결과로 세지 않았다.

기존 결과의 실제 영향, 전체 report 재빌드, 실제 장비 비교는 후속 점검 범위다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| CFAR 누적합과 동적 범위 회귀검사 | 기존 검출 원장을 재계산할 범위를 결정한다 | `src/detection_gpu.py` |
| 수신 처리의 truth 분리와 프레임별 reference 지원 | 거리·방향·추적 성능 평가에 쓸 입력 계약을 결정한다 | `src/experiment_detection.py` · `src/passive_process.py` |
| 설치 조합 기록과 CPU smoke 실행 경로 정리 | 새 checkout에서 재현 가능한 최소 실행 경로를 결정한다 | `runners/SOLVER_BUILDS.json` · `docs/REPRODUCE.md` |
| 최신 상태 입구와 보고서 빌드 문제 정리 | 다음 실험에 쓸 문서·원장·빌더의 기준을 결정한다 | `docs/RESUME.md` · `docs/MESH_GALLERY_STALE_0914.md` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 30개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/directory_review_0915.json` | `gate_count` | 6 |
| [^2] | `outputs/directory_review_0915.json` | `gate_pass_count` | 6 |
| [^3] | `outputs/directory_review_0915.json` | `volumes.n_volumes` | 13 |
| [^4] | `outputs/directory_review_0915.json` | `volumes.n_companions` | 7 |
| [^5] | `outputs/directory_review_0915.json` | `volumes.n_notebooks` | 33 |
| [^6] | `outputs/directory_review_0915.json` | `cfar.setup.shape[0]` | 64 |
| [^7] | `outputs/directory_review_0915.json` | `cfar.setup.shape[1]` | 32 |
| [^8] | `outputs/directory_review_0915.json` | `cfar.setup.pfa_nominal` | 0.0001 |
| [^9] | `outputs/directory_review_0915.json` | `cfar.setup.strong_cell[0]` | 32 |
| [^10] | `outputs/directory_review_0915.json` | `cfar.setup.strong_cell[1]` | 2 |
| [^11] | `outputs/directory_review_0915.json` | `cfar.rows[4].peak_over_unit_background_power_db` | 100 |
| [^12] | `outputs/directory_review_0915.json` | `cfar.rows[4].torch32_off_guard_hits` | 583 |
| [^13] | `outputs/directory_review_0915.json` | `cfar.rows[4].torch64_off_guard_hits` | 0 |
| [^14] | `outputs/directory_review_0915.json` | `range_and_steering.same_IQ_different_ground_truth_range.truth_ranges_m[0]` | 200 |
| [^15] | `outputs/directory_review_0915.json` | `range_and_steering.same_IQ_different_ground_truth_range.truth_ranges_m[1]` | 300 |
| [^16] | `outputs/directory_review_0915.json` | `range_and_steering.same_IQ_different_ground_truth_range.axis_shift_min_m` | 100 |
| [^17] | `outputs/directory_review_0915.json` | `reference_frames.setup.frames` | 32 |
| [^18] | `outputs/directory_review_0915.json` | `reference_frames.setup.length` | 128 |
| [^19] | `outputs/directory_review_0915.json` | `reference_frames.setup.delay` | 7 |
| [^20] | `outputs/directory_review_0915.json` | `reference_frames.setup.doppler_offset` | 5 |
| [^21] | `outputs/directory_review_0915.json` | `reference_frames.rows[1].target_cell_ratio_db` | -37.78 |
| [^22] | `outputs/directory_review_0915.json` | `reference_frames.rows[0].target_cell_ratio_db` | 0 |
| [^23] | `outputs/directory_review_0915.json` | `syntax.files` | 749 |
| [^24] | `outputs/directory_review_0915.json` | `syntax.error_count` | 0 |
| [^25] | `outputs/directory_review_0915.json` | `gates.check_reader_gate.exit_code` | 0 |
| [^26] | `outputs/directory_review_0915.json` | `gates.check_report_links.exit_code` | 0 |
| [^27] | `outputs/directory_review_0915.json` | `gates.check_retracted.exit_code` | 0 |
| [^28] | `outputs/directory_review_0915.json` | `gates.check_stale_titles.exit_code` | 0 |
| [^29] | `outputs/directory_review_0915.json` | `gates.check_row_pointers.exit_code` | 0 |
| [^30] | `outputs/directory_review_0915.json` | `gates.check_new_file_rules.exit_code` | 0 |